In [1]:
from src.graph.orchastrator import HierarchicalAgent
from src.graph.orchastrator.preprocessing import prepare_orchestrator_input
from src.database.data_preprocessor import preprocess_user

/Users/rogier/dev/joho/advies_agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Preprocess a user by aanvraag_id
user = preprocess_user(118)
inputs = prepare_orchestrator_input(user)

print(f"Providers: {inputs['available_providers']}")
print(f"Product descriptions loaded: {list(inputs['product_descriptions'].keys())}")
print(f"User profile keys: {list(inputs['user_profile'].keys())}")

Providers: ['oom_wib', 'allianz_globetrotter', 'goudse_expat_pakket', 'oom_tib', 'goudse_ngo_zendelingen', 'special_isis', 'globality_yougenio', 'expatriate_group', 'International Expat Insurance', 'goudse_working_nomad']
Product descriptions loaded: ['oom_wib', 'allianz_globetrotter', 'goudse_expat_pakket', 'oom_tib', 'goudse_ngo_zendelingen', 'special_isis', 'globality_yougenio', 'expatriate_group', 'goudse_working_nomad']
User profile keys: ['aanvraag_id', 'family', 'hoofdreden_verblijf', 'toelichting_hoofdreden', 'verwachte_duur_verblijf', 'toelichting_duur', 'functieomschrijving', 'type_werkzaamheden', 'interesse_internationale_aov', 'interesse_zkv', 'sporten_activiteiten', 'sport_semiprofessioneel', 'eerder_contact_joho', 'eerder_contact_keuze', 'advies_vorm']


In [3]:
inputs

{'user_profile': {'aanvraag_id': 118,
  'family': [{'role': 'main', 'age': 34, 'birth_date': '1992-02-06'}],
  'hoofdreden_verblijf': 'Werken voor eigen onderneming',
  'toelichting_hoofdreden': 'Ik heb bij de de Goudse verzekeringen een verzekering via jullie met polisnunmer 273417. Graag hier een aansprakelijksverzekering bij voor werk in Europa. Resident en zzper in Albanie.',
  'verwachte_duur_verblijf': 'Permanent',
  'toelichting_duur': 'Ik wil graag een aansprakelijkheidsverzekering. Ik ben zzper in Albanie. Werk vooral in Europa. ',
  'functieomschrijving': 'Mud-engineer in HDD drilling. Werk in Polen op het moment.',
  'type_werkzaamheden': 'Middelzwaar tot zwaar werk',
  'interesse_internationale_aov': 'Ja - ik ben geïnteresseerd in informatie over voordeligere opties om met name voor de grote risico’s (ernstige ziekte en zware ongevallen) verzekerd te zijn',
  'interesse_zkv': 'Nee',
  'sporten_activiteiten': 'Hardlopen ',
  'sport_semiprofessioneel': 'Nee',
  'eerder_contac

In [4]:
agent = HierarchicalAgent()

In [5]:
# Stream execution — see each node as it completes
for event in agent.stream(**inputs):
    node_name = list(event.keys())[0]
    updates = event[node_name]
    print(f"\n{'='*60}")
    print(f"Node: {node_name}")
    print(f"Returned keys: {list(updates.keys())}")

    # Show key info per node
    if node_name == "user_agent":
        pc = updates.get("parsed_constraints", {})
        aspects = pc.get("retrieval_aspects", [])
        print(f"  Retrieval aspects: {[a['aspect'] for a in aspects]}")
        print(f"  Interpreted needs: {pc.get('interpreted_needs', [])}")

    elif node_name == "build_tracker":
        tracker = updates.get("retrieval_tracker", {})
        for p, data in tracker.items():
            levels = list(data.get("coverage_levels", {}).keys())
            print(f"  {p}: {len(data['aspects'])} aspects, levels={levels}")

    elif node_name == "orchestrator_assess":
        tasks = updates.get("retrieval_tasks", [])
        tracker = updates.get("retrieval_tracker", {})
        dropped = [p for p, d in tracker.items() if d.get("status") == "dropped"]
        print(f"  Tasks dispatched: {len(tasks)}")
        for t in tasks:
            print(f"    -> {t['provider']}: {t['aspect']} | {t['query'][:80]}")
        if dropped:
            print(f"  Dropped providers: {dropped}")

    elif node_name == "retriever":
        summaries = updates.get("retrieval_summaries", [])
        for s in summaries:
            if s:
                print(f"  {s.get('provider')}/{s.get('aspect')}: confidence={s.get('confidence')}")
                print(f"    {s.get('overall_summary', '')[:120]}...")

    elif node_name == "merge_results":
        print(f"  Iteration: {updates.get('retrieval_iteration')}")

    elif node_name == "evaluator_step1":
        qa = updates.get("qualitative_assessment", {})
        assessments = qa.get("assessments", [])
        for a in assessments:
            print(f"  {a['provider']} {a['coverage_level']}: €{a['premium']}")
            print(f"    Fit: {a['overall_fit'][:100]}...")

    elif node_name == "evaluator_step2":
        rec = updates.get("recommendation", {})
        print("  TOP RECOMMENDATIONS:")
        for r in rec.get("top_recommendations", []):
            print(f"    {r['provider']} {r['coverage_level']} (€{r['premium']})")
        print("  BUDGET RECOMMENDATIONS:")
        for r in rec.get("budget_recommendations", []):
            print(f"    {r['provider']} {r['coverage_level']} (€{r['premium']})")

TypeError: "Could not resolve authentication method. Expected either api_key or auth_token to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"

In [ ]:
# Or run without streaming to get the final state
# result = agent.invoke(**inputs)
# result["recommendation"]